# OpenPlaque — RCA PCAT radial-boundary diagnostic

This notebook keeps the **accepted RCA centerline and lumen-radius model fixed** and asks a narrower question:

> **Why does measured PCAT attenuation become less negative farther from the artery?**

It uses the current **0.75 mm approximate outer-wall offset**, then analyzes the 10–50 mm RCA segment in **0.5 mm radial layers** outward from that modeled wall.

For each radial layer it reports:
- all candidate shell voxels
- adipose voxels in the analysis range **−190 to −30 HU**
- voxels **> −30 HU** (likely non-fat / soft tissue / partial-volume contamination)
- voxels **< −190 HU** (air/very-low attenuation)
- adipose fraction
- mean/median PCAT attenuation
- a restricted **0–3 mm** PCAT result

It also creates radial × longitudinal QC heatmaps and cross-sections at 10/20/30/40/50 mm.

This is an OpenPlaque research prototype, **not Caristo FAI-Score**.


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch pcat-radial-boundary-diagnostic-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy matplotlib pandas
print('Repository and packages ready.')


In [ ]:
import sys, shutil, zipfile, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import ndimage as ndi
from scipy.spatial import cKDTree

sys.path.insert(0, '/content/OpenPlaque/src')
from openplaque.study import OpenPlaqueStudy

ROOT = Path('/content/drive/MyDrive/OpenPlaque')
BASE = ROOT/'PCAT_RCA_10_50'
OUT = ROOT/'PCAT_RCA_10_50_Radial_Diagnostic'
OUT.mkdir(parents=True, exist_ok=True)

WALL_MARGIN_MM = 0.75
FAT_LO_HU, FAT_HI_HU = -190.0, -30.0
SEGMENT_START_MM, SEGMENT_END_MM = 10.0, 50.0
RADIAL_BIN_MM = 0.5
RADIAL_MAX_MM = 6.0
RESTRICTED_MAX_MM = 3.0

print('Output:', OUT)


## 1. Load source CCTA and fixed centerline/radius inputs

The centerline and lumen-radius profile are **not recalculated** here.


In [ ]:
DRIVE_ZIP = ROOT/'Full_DICOM.zip'
LOCAL_ZIP = Path('/content/Full_DICOM.zip')
EXTRACT_ROOT = '/content/full_dicom_pcat_radial_diag'

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(DRIVE_ZIP)
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)

shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
study = OpenPlaqueStudy(str(LOCAL_ZIP), extract_root=EXTRACT_ROOT)
source_img, ct, _ = study.load_series(7)
ct = np.asarray(ct)
sp_xyz = np.array(source_img.GetSpacing(), float)
sp_zyx = sp_xyz[::-1]
voxel_mm3 = float(np.prod(sp_xyz))

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

centerline_path = first_existing([
    BASE/'rca_centerline_smoothed_zyx.csv',
    *ROOT.rglob('rca_centerline_smoothed_zyx.csv')
])
radius_path = first_existing([
    BASE/'pcat_local_radius_profile.csv',
    *ROOT.rglob('pcat_local_radius_profile.csv')
])

if centerline_path is None or radius_path is None:
    raise FileNotFoundError('Run RCA_PCAT_10_50_Prototype.ipynb first.')

cl = pd.read_csv(centerline_path)
rad = pd.read_csv(radius_path)

lumen_r_all = np.interp(
    cl.arc_mm.to_numpy(float),
    rad.arc_mm.to_numpy(float),
    rad.lumen_radius_mm.to_numpy(float)
)

arc_all = cl.arc_mm.to_numpy(float)
seg_mask = (arc_all >= SEGMENT_START_MM) & (arc_all <= SEGMENT_END_MM)
seg = cl.loc[seg_mask].reset_index(drop=True)
seg_arc = seg.arc_mm.to_numpy(float)
seg_zyx = seg[['z','y','x']].to_numpy(float)
seg_mm = seg_zyx * sp_zyx
seg_lumen_r = lumen_r_all[seg_mask]
seg_outer_r = seg_lumen_r + WALL_MARGIN_MM
seg_shell_outer_r = seg_outer_r * 3.0

print('CT shape:', ct.shape)
print('Spacing xyz mm:', tuple(sp_xyz))
print('Centerline:', centerline_path)
print('Radius profile:', radius_path)
print('Segment points:', len(seg))
print('Mean lumen radius:', round(float(seg_lumen_r.mean()),3), 'mm')
print('Mean modeled outer radius:', round(float(seg_outer_r.mean()),3), 'mm')


## 2. Build one physical crop and nearest-centerline coordinate map


In [ ]:
pad_mm = float(np.max(seg_shell_outer_r) + 3.0)
lo = np.floor(np.min(seg_zyx, axis=0) - pad_mm/sp_zyx).astype(int)
hi = np.ceil(np.max(seg_zyx, axis=0) + pad_mm/sp_zyx).astype(int) + 1
lo = np.maximum(lo, 0)
hi = np.minimum(hi, np.array(ct.shape))

crop = ct[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
zz,yy,xx = np.indices(crop.shape)
glob_zyx = np.stack([zz+lo[0], yy+lo[1], xx+lo[2]], axis=-1).reshape(-1,3).astype(float)
glob_mm = glob_zyx * sp_zyx

tree = cKDTree(seg_mm)
dist_mm, nearest_idx = tree.query(glob_mm, k=1, workers=-1)
nearest_idx = nearest_idx.astype(int)
nearest_arc = seg_arc[nearest_idx]
nearest_lumen = seg_lumen_r[nearest_idx]
nearest_outer = seg_outer_r[nearest_idx]
nearest_shell_outer = seg_shell_outer_r[nearest_idx]
radial_outward = dist_mm - nearest_outer
hu_flat = crop.reshape(-1).astype(float)

aorta_candidates = [
    ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz',
    ROOT/'TotalSegmentator_Validation_v2'/'aorta_series7_totalseg.nii.gz',
]
ap = next((p for p in aorta_candidates if p.exists()), None)
if ap is not None:
    ai = sitk.ReadImage(str(ap))
    if ai.GetSize()!=source_img.GetSize() or not np.allclose(ai.GetSpacing(), source_img.GetSpacing()):
        ai = sitk.Resample(ai, source_img, sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)
    aorta = sitk.GetArrayFromImage(ai)>0
    aorta_crop = aorta[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]].reshape(-1)
else:
    aorta_crop = np.zeros_like(hu_flat, dtype=bool)

shell = (
    (dist_mm > nearest_outer) &
    (dist_mm <= nearest_shell_outer) &
    (nearest_arc >= SEGMENT_START_MM) &
    (nearest_arc <= SEGMENT_END_MM) &
    (~aorta_crop)
)
fat = shell & (hu_flat >= FAT_LO_HU) & (hu_flat <= FAT_HI_HU)
high = shell & (hu_flat > FAT_HI_HU)
low = shell & (hu_flat < FAT_LO_HU)

print('Crop shape:', crop.shape)
print('Shell voxels:', int(shell.sum()))
print('Fat voxels:', int(fat.sum()))
print('High-HU/non-fat voxels:', int(high.sum()))
print('Very-low-HU voxels:', int(low.sum()))


## 3. Fine 0.5-mm radial composition analysis


In [ ]:
edges = np.arange(0.0, RADIAL_MAX_MM + RADIAL_BIN_MM + 1e-9, RADIAL_BIN_MM)
rad_rows = []

for a,b in zip(edges[:-1], edges[1:]):
    candidate = shell & (radial_outward >= a) & (radial_outward < b)
    fv = candidate & (hu_flat >= FAT_LO_HU) & (hu_flat <= FAT_HI_HU)
    hv = candidate & (hu_flat > FAT_HI_HU)
    lv = candidate & (hu_flat < FAT_LO_HU)
    vals = hu_flat[fv]
    n = int(candidate.sum())
    rad_rows.append({
        'radial_start_mm': float(a),
        'radial_end_mm': float(b),
        'candidate_voxels': n,
        'fat_voxels': int(fv.sum()),
        'high_gt_minus30_voxels': int(hv.sum()),
        'low_lt_minus190_voxels': int(lv.sum()),
        'fat_fraction': float(fv.sum()/n) if n else np.nan,
        'high_gt_minus30_fraction': float(hv.sum()/n) if n else np.nan,
        'low_lt_minus190_fraction': float(lv.sum()/n) if n else np.nan,
        'pcat_mean_hu': float(np.mean(vals)) if len(vals) else np.nan,
        'pcat_median_hu': float(np.median(vals)) if len(vals) else np.nan,
        'pcat_sd_hu': float(np.std(vals)) if len(vals) else np.nan,
    })

radial = pd.DataFrame(rad_rows)
radial.to_csv(OUT/'pcat_radial_0p5mm_composition.csv', index=False)
display(radial)


## 4. Restricted 0–3 mm analysis

This asks whether the unexpected radial trend persists in the better-supported, more vessel-proximal region.


In [ ]:
restricted = fat & (radial_outward >= 0) & (radial_outward < RESTRICTED_MAX_MM)
rvals = hu_flat[restricted]

r0 = radial[radial.radial_end_mm <= RESTRICTED_MAX_MM].copy()
good = r0.fat_voxels >= 100
if good.sum() >= 3:
    mids = (r0.loc[good,'radial_start_mm'].to_numpy(float) + r0.loc[good,'radial_end_mm'].to_numpy(float))/2
    means = r0.loc[good,'pcat_mean_hu'].to_numpy(float)
    slope03 = float(np.polyfit(mids, means, 1)[0])
else:
    slope03 = np.nan

summary = pd.DataFrame([{
    'wall_margin_mm': WALL_MARGIN_MM,
    'segment_start_mm': SEGMENT_START_MM,
    'segment_end_mm': SEGMENT_END_MM,
    'full_shell_pcat_mean_hu': float(np.mean(hu_flat[fat])),
    'restricted_0_3mm_pcat_mean_hu': float(np.mean(rvals)),
    'restricted_0_3mm_pcat_median_hu': float(np.median(rvals)),
    'restricted_0_3mm_pcat_sd_hu': float(np.std(rvals)),
    'restricted_0_3mm_fat_voxels': int(len(rvals)),
    'restricted_0_3mm_fat_volume_ml': float(len(rvals)*voxel_mm3/1000),
    'restricted_0_3mm_radial_slope_hu_per_mm': slope03,
    'overall_shell_high_gt_minus30_fraction': float(high.sum()/max(1,shell.sum())),
    'overall_shell_low_lt_minus190_fraction': float(low.sum()/max(1,shell.sum())),
}])
summary.to_csv(OUT/'pcat_radial_boundary_summary.csv', index=False)
display(summary.T)


## 5. Longitudinal × radial composition matrices


In [ ]:
arc_bins = np.arange(10, 51, 1.0)
rad_bins = np.arange(0, RADIAL_MAX_MM + RADIAL_BIN_MM, RADIAL_BIN_MM)

rows=[]
for aa,bb in zip(arc_bins[:-1],arc_bins[1:]):
    for rr0,rr1 in zip(rad_bins[:-1],rad_bins[1:]):
        candidate = shell & (nearest_arc>=aa) & (nearest_arc<bb) & (radial_outward>=rr0) & (radial_outward<rr1)
        if candidate.sum()==0:
            continue
        fv = candidate & (hu_flat>=FAT_LO_HU) & (hu_flat<=FAT_HI_HU)
        hv = candidate & (hu_flat>FAT_HI_HU)
        vals=hu_flat[fv]
        rows.append({
            'arc_start_mm':float(aa),'arc_end_mm':float(bb),
            'radial_start_mm':float(rr0),'radial_end_mm':float(rr1),
            'candidate_voxels':int(candidate.sum()),
            'fat_voxels':int(fv.sum()),
            'high_gt_minus30_fraction':float(hv.sum()/candidate.sum()),
            'fat_fraction':float(fv.sum()/candidate.sum()),
            'pcat_mean_hu':float(np.mean(vals)) if len(vals) else np.nan,
        })
grid = pd.DataFrame(rows)
grid.to_csv(OUT/'pcat_radial_longitudinal_grid.csv', index=False)
print('Grid rows:', len(grid))


## 6. Figures


In [ ]:
# Figure 1: radial composition + PCAT
fig, axs = plt.subplots(1,3,figsize=(16,4.5))
mids=(radial.radial_start_mm+radial.radial_end_mm)/2
axs[0].plot(mids, radial.pcat_mean_hu, marker='o')
axs[0].axvline(3.0,ls='--')
axs[0].set(xlabel='mm outward from modeled outer wall',ylabel='PCAT mean HU',title='0.5-mm radial PCAT')
axs[1].plot(mids, radial.fat_fraction, marker='o', label='fat fraction')
axs[1].plot(mids, radial.high_gt_minus30_fraction, marker='o', label='> -30 HU fraction')
axs[1].plot(mids, radial.low_lt_minus190_fraction, marker='o', label='< -190 HU fraction')
axs[1].axvline(3.0,ls='--')
axs[1].set(xlabel='mm outward',ylabel='fraction of candidate shell voxels',title='Radial tissue composition')
axs[1].legend()
axs[2].bar(mids, radial.candidate_voxels, width=0.4, label='candidate')
axs[2].bar(mids, radial.fat_voxels, width=0.4, label='fat')
axs[2].axvline(3.0,ls='--')
axs[2].set(xlabel='mm outward',ylabel='voxels',title='Radial support')
axs[2].legend()
plt.tight_layout()
p1=OUT/'01_radial_boundary_diagnostic.png'
fig.savefig(p1,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


In [ ]:
# Figure 2: radial x longitudinal heatmaps
def pivot_val(col):
    q=grid.copy()
    q['arc_mid']=(q.arc_start_mm+q.arc_end_mm)/2
    q['rad_mid']=(q.radial_start_mm+q.radial_end_mm)/2
    return q.pivot(index='rad_mid',columns='arc_mid',values=col)

fig,axs=plt.subplots(1,3,figsize=(17,5))
for ax,col,title in [
    (axs[0],'pcat_mean_hu','PCAT mean HU'),
    (axs[1],'fat_fraction','fat fraction'),
    (axs[2],'high_gt_minus30_fraction','> -30 HU fraction'),
]:
    P=pivot_val(col)
    im=ax.imshow(P.values,aspect='auto',origin='lower',
                 extent=[P.columns.min(),P.columns.max(),P.index.min(),P.index.max()])
    ax.set(xlabel='arc from ostium (mm)',ylabel='mm outward',title=title)
    fig.colorbar(im,ax=ax)
plt.tight_layout()
p2=OUT/'02_radial_longitudinal_heatmaps.png'
fig.savefig(p2,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


In [ ]:
# Figure 3: perpendicular-plane local diagnostics at 10/20/30/40/50 mm
# Visualized on axial slices for robust QC: modeled lumen/outer-wall/shell radii are drawn
# around the fixed centerline point using physical in-plane spacing.
marks=[10,20,30,40,50]
fig,axs=plt.subplots(1,5,figsize=(18,4))
for ax,s0 in zip(axs,marks):
    i=int(np.argmin(np.abs(seg_arc-s0)))
    z,y,x=seg_zyx[i]
    zi=int(round(z))
    span_mm=10
    hy=int(math.ceil(span_mm/sp_zyx[1])); hx=int(math.ceil(span_mm/sp_zyx[2]))
    y0=max(0,int(round(y))-hy); y1=min(ct.shape[1],int(round(y))+hy+1)
    x0=max(0,int(round(x))-hx); x1=min(ct.shape[2],int(round(x))+hx+1)
    ax.imshow(ct[zi,y0:y1,x0:x1],cmap='gray',vmin=-200,vmax=500)
    cx=x-x0; cy=y-y0
    ax.plot(cx,cy,'o',ms=4)
    # Approximate circles in axial plane for visual QC only.
    for r in [seg_lumen_r[i],seg_outer_r[i],seg_shell_outer_r[i]]:
        th=np.linspace(0,2*np.pi,200)
        ax.plot(cx+(r/sp_zyx[2])*np.cos(th),cy+(r/sp_zyx[1])*np.sin(th),lw=1)
    ax.set_title(f'{s0} mm, z={zi}')
    ax.axis('off')
plt.tight_layout()
p3=OUT/'03_crosssection_geometry_qc.png'
fig.savefig(p3,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)


## 7. Package everything to report back


In [ ]:
files_to_zip=[
    '01_radial_boundary_diagnostic.png',
    '02_radial_longitudinal_heatmaps.png',
    '03_crosssection_geometry_qc.png',
    'pcat_radial_0p5mm_composition.csv',
    'pcat_radial_longitudinal_grid.csv',
    'pcat_radial_boundary_summary.csv',
]
zip_path=OUT/'PCAT_RADIAL_BOUNDARY_REPORT_BACK.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for name in files_to_zip:
        p=OUT/name
        if p.exists():
            zf.write(p,arcname=name)

print('ZIP:', zip_path)
print('\nDirect Drive search links:')
for name in [zip_path.name]+files_to_zip:
    print(f'https://drive.google.com/drive/u/0/search?q={name}')
